# Imports

In [9]:
import numpy as np
import pandas as pd
import category_encoders as ce

from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from boruta import BorutaPy

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector, SelectFromModel, RFECV

from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

## Utils

In [10]:
from __future__ import annotations

from typing import Iterable, Sequence
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class FeatureSelector(BaseEstimator, TransformerMixin):
    """Select a subset of features from a DataFrame or NumPy array.

    Parameters
    ----------
    features : Iterable[str] or Iterable[int], default=None
        List of column names (for pandas) or column indices (for numpy/pandas) 
        to select.

    mask : Iterable[bool], default=None
        Boolean mask of length n_features_in_ indicating which features to select.

    check_missing : bool, default=True
        If True, validates whether the requested features exist in the input data.
        Only applicable when `features` are provided as strings.

    Attributes
    ----------
    selected_features_ : list
        List of selected feature names (strings) or indices (ints).

    n_features_in_ : int
        Number of features seen during `fit`.

    feature_names_in_ : ndarray of shape (n_features_in_,)
        Names of features seen during `fit`. Defined only when `X` has feature
        names (e.g., a pandas DataFrame) or generated as string indices for numpy.
    """

    def __init__(
        self, 
        features: Iterable[str | int] | None = None, 
        mask: Iterable[bool] | None = None, 
        check_missing: bool = True
    ):
        self.features = features
        self.mask = mask
        self.check_missing = check_missing

    def fit(self, X, y=None):
        """Fit the feature selector to the input data.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            The training input samples.
        y : None
            Ignored. This parameter exists only for compatibility with 
            sklearn's Pipeline.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        # Validate mutual exclusivity of parameters
        if self.features is None and self.mask is None:
            raise ValueError("Either 'features' or 'mask' must be provided.")
        if self.features is not None and self.mask is not None:
            raise ValueError("Only one of 'features' or 'mask' should be provided.")

        # Determine input type and extract metadata
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
            self.n_features_in_ = X.shape[1]
            is_pandas = True
        else:
            X_arr = np.asarray(X)
            self.n_features_in_ = X_arr.shape[1]
            # Generate synthetic names for numpy alignment
            self.feature_names_in_ = np.array([f"x{i}" for i in range(self.n_features_in_)], dtype=object)
            is_pandas = False

        # Case 1: Mask-based selection
        if self.mask is not None:
            mask = np.asarray(self.mask, dtype=bool)
            if len(mask) != self.n_features_in_:
                raise ValueError(
                    f"Mask length ({len(mask)}) does not match "
                    f"number of features ({self.n_features_in_})."
                )
            
            # Save selection based on input type
            if is_pandas:
                self.selected_features_ = list(self.feature_names_in_[mask])
            else:
                self.selected_features_ = list(np.where(mask)[0])

        # Case 2: Explicit features list (names or indices)
        else:
            features_list = list(self.features)
            
            # If strings are passed but input is numpy, try to map from synthetic names or indices
            if not is_pandas and all(isinstance(f, str) for f in features_list):
                # If they passed synthetic names like ['x0', 'x2']
                if all(f in self.feature_names_in_ for f in features_list):
                    self.selected_features_ = [list(self.feature_names_in_).index(f) for f in features_list]
                else:
                    raise ValueError("String feature names cannot be mapped to a NumPy array unless they match 'x0', 'x1', etc.")
            else:
                self.selected_features_ = features_list

            # Optional check for missing columns (only makes sense for string names in pandas)
            if self.check_missing and is_pandas and all(isinstance(f, str) for f in self.selected_features_):
                missing = sorted(set(self.selected_features_) - set(self.feature_names_in_))
                if missing:
                    raise ValueError(f"The following features do not exist in X: {missing}")

        return self

    def transform(self, X):
        """Reduce X to the selected features.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            The input samples.

        Returns
        -------
        X_sliced : {array-like, sparse matrix} of shape (n_samples, n_selected_features)
            The input samples with only the selected features.
        """
        check_is_fitted(self, attributes=["selected_features_", "n_features_in_"])

        if isinstance(X, pd.DataFrame):
            # If fit was on pandas or indices are used, pandas .loc/.iloc handles it safely
            if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
                return X.iloc[:, self.selected_features_]
            return X.loc[:, self.selected_features_]
        else:
            X_arr = np.asarray(X)
            # If selected_features_ contains string names (from pandas fit) but X is numpy
            if Janus_indices := [isinstance(f, str) for f in self.selected_features_]:
                if any(Janus_indices):
                    # Map string names back to positions using stored feature_names_in_
                    indices = [list(self.feature_names_in_).index(f) for f in self.selected_features_]
                    return X_arr[:, indices]
            
            return X_arr[:, self.selected_features_]

    def inverse_transform(self, X):
        """Reverse the transformation, filling unselected features with NaNs.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_selected_features)
            The converted input samples.

        Returns
        -------
        X_original : {array-like, sparse matrix} of shape (n_samples, n_features_in_)
            The original structure filled with NaNs where features were excluded.
        """
        check_is_fitted(self, attributes=["selected_features_", "feature_names_in_"])

        if isinstance(X, pd.DataFrame):
            out = pd.DataFrame(index=X.index, columns=self.feature_names_in_)
            if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
                cols = self.feature_names_in_[self.selected_features_]
                out[cols] = X.values
            else:
                out[self.selected_features_] = X
            return out
        else:
            X_arr = np.asarray(X)
            out = np.full((X_arr.shape[0], self.n_features_in_), np.nan, dtype=float)
            
            # Map features to numeric indices for array assignment
            if all(isinstance(f, str) for f in self.selected_features_):
                indices = [list(self.feature_names_in_).index(f) for f in self.selected_features_]
            else:
                indices = self.selected_features_
                
            out[:, indices] = X_arr
            return out

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        """Get output feature names for transformation."""
        check_is_fitted(self, attributes=["selected_features_"])
        if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
            return np.asarray(self.feature_names_in_[self.selected_features_], dtype=object)
        return np.asarray(self.selected_features_, dtype=object)

    def get_support(self, indices: bool = False) -> np.ndarray:
        """Get a mask or index array of the features selected."""
        check_is_fitted(self, attributes=["selected_features_", "feature_names_in_"])
        
        if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
            mask = np.zeros(self.n_features_in_, dtype=bool)
            mask[self.selected_features_] = True
        else:
            mask = np.isin(self.feature_names_in_, self.selected_features_)

        if indices:
            return np.where(mask)[0]
        return mask

    @property
    def features_(self) -> list:
        """Backward compatibility helper for selected features."""
        check_is_fitted(self, attributes=["selected_features_"])
        return self.selected_features_

    def __len__(self) -> int:
        check_is_fitted(self, attributes=["selected_features_"])
        return len(self.selected_features_)

    def __repr__(self) -> str:
        try:
            length = len(self)
        except Exception:
            length = "Not Fitted"
        return f"FeatureSelector(n_features={length})"

# Loading Dataset

In [11]:
X_train_raw = pd.read_parquet('../data/X_train_raw.parquet')
X_train_fe = pd.read_parquet('../data/X_train_fe.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test_raw = pd.read_parquet('../data/X_test_raw.parquet')
X_test_fe = pd.read_parquet('../data/X_test_fe.parquet')

In [19]:
X_train_raw['spectral_type'] = X_train_raw['spectral_type'].astype('category')
X_train_raw['galaxy_population'] = X_train_raw['galaxy_population'].astype('category')

X_test_raw['spectral_type'] = X_test_raw['spectral_type'].astype('category')
X_test_raw['galaxy_population'] = X_test_raw['galaxy_population'].astype('category')

In [20]:
X_train_fe['spectral_type'] = X_train_fe['spectral_type'].astype('category')
X_train_fe['galaxy_population'] = X_train_fe['galaxy_population'].astype('category')

X_test_fe['spectral_type'] = X_test_fe['spectral_type'].astype('category')
X_test_fe['galaxy_population'] = X_test_fe['galaxy_population'].astype('category')

In [21]:
X_train_raw.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


In [22]:
X_train_fe.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,...,1.537632,1.100813,0.636056,5.114196,6.851065,8.326031,7.875818,0.313090,0.024912,-0.949397
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,...,1.499854,0.361141,0.439634,3.191300,3.992076,2.778353,2.721302,-0.408896,0.435262,0.802092
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,...,-0.092712,0.589211,0.025263,-0.136637,0.477837,59.784407,58.120611,0.529713,0.466346,-0.708467
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,...,2.032982,0.652096,0.450706,4.287302,5.390104,10.195392,9.845805,-0.116193,0.045921,-0.992165
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,...,1.237231,0.335002,0.283262,3.468709,4.086973,10.134002,9.947821,-0.787468,-0.394539,0.473532


In [23]:
X_test_raw.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [24]:
X_test_fe.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence,...,0.865497,0.906151,0.977908,2.581883,4.465942,9.046867,8.658089,0.081333,0.344971,-0.935083
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence,...,1.606658,0.871833,0.172640,4.170770,5.215243,17.981106,17.224961,-0.208809,0.113279,-0.971374
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud,...,0.923319,0.980640,0.393909,1.876277,3.250826,3.935608,3.715715,0.131045,0.129932,-0.982825
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence,...,0.856696,0.252526,-0.281462,2.450870,2.421934,1.374717,1.357922,-0.143212,0.069175,-0.987272
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence,...,1.520808,1.683673,0.345342,3.971109,6.000123,20.641941,18.996626,0.799820,-0.507330,0.320787


In [25]:
y_train.head()

,class,class_encoded
id,,
0,GALAXY,0
1,GALAXY,0
2,QSO,1
3,GALAXY,0
4,GALAXY,0


# Feature Selection

## Base Model

In [33]:
lgbm = LGBMClassifier(
    objective="multiclass",
    metric="multi_logloss",
    num_class=3,
    boosting_type="gbdt",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    feature_fraction=0.85,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=1,
    enable_categorical=True
)

cat = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False,
    thread_count=1,
)

models = {
    'lgbm': lgbm,
    'xgb': xgb,
    'cat': cat,
}

In [36]:
X_train_raw.dtypes

alpha                 float64
delta                 float64
u                     float64
g                     float64
r                     float64
i                     float64
z                     float64
redshift              float64
spectral_type        category
galaxy_population    category
dtype: object

In [38]:
for model_str, model in tqdm(models.items()):
    
    cv_result = cross_val_score(
        model, 
        X_train_raw, 
        y_train.class_encoded, 
        scoring='neg_log_loss', 
        cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
        params={'cat_features': ['spectral_type', 'galaxy_population']} if model_str == 'cat' else None 
    ).mean()

    print(f"{model_str}: {cv_result}")

 33%|██████████████████████████████████████████████████████████▋                                                                                                                     | 1/3 [07:36<15:13, 456.52s/it]

lgbm: -0.08893223167393138


 67%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                          | 2/3 [14:43<07:19, 439.01s/it]

xgb: -0.0906569586517374


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [47:03<00:00, 941.31s/it]

cat: -0.10162354118001204


In [39]:
for model_str, model in tqdm(models.items()):
    
    cv_result = cross_val_score(
        model, 
        X_train_fe, 
        y_train.class_encoded, 
        scoring='neg_log_loss', 
        cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
        params={'cat_features': ['spectral_type', 'galaxy_population']} if model_str == 'cat' else None 
    ).mean()

    print(f"{model_str}: {cv_result}")

 33%|██████████████████████████████████████████████████████████▋                                                                                                                     | 1/3 [10:02<20:04, 602.29s/it]

lgbm: -0.08971049761891746


 67%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                          | 2/3 [19:03<09:26, 566.56s/it]

xgb: -0.09001558318100678


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [55:19<00:00, 1106.63s/it]

cat: -0.09782820290665671
